# Goal: Recreate Gapminder's Bubble Chart*

![Gapminder](Gapminder.png)

*As much as possible...

See https://www.gapminder.org/tools/

## Import libraries

In [21]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objs as go

In [22]:
import plotly
print(plotly.__version__)

6.7.0


## Load the data

In [23]:
df = pd.read_csv('Gapminder-data.csv', sep=',')

In [24]:
df.head()

,Country,Region,Year,Income,Life expectancy,Population,Fertility,Child mortality
0,Afghanistan,Asia,1900,1088.0,29.41,4832414.0,7.001,481.77
1,Afghanistan,Asia,1901,1106.0,29.47,4879685.0,7.001,481.36
2,Afghanistan,Asia,1902,1124.0,29.53,4935122.0,7.001,480.87
3,Afghanistan,Asia,1903,1143.0,29.60,4998861.0,7.001,480.30
4,Afghanistan,Asia,1904,1162.0,29.66,5063419.0,7.001,481.08


In [25]:
df.describe()

,Year,Income,Life expectancy,Population,Fertility,Child mortality
count,22424.000000,22424.000000,22424.000000,2.242400e+04,22424.000000,22424.000000
mean,1960.114877,7759.233500,52.851484,2.023149e+07,4.807106,189.525802
std,34.893923,13587.506024,16.849974,8.630455e+07,1.931171,152.436433
min,1900.000000,312.000000,1.100000,1.640900e+04,0.877000,1.580000
25%,1930.000000,1385.000000,35.820000,8.617980e+05,2.910000,45.807500
50%,1960.000000,2915.500000,53.910000,3.561738e+06,5.392000,157.185000
75%,1990.000000,7885.250000,68.532500,1.054941e+07,6.500000,328.532500
max,2020.000000,178635.000000,85.290000,1.439324e+09,9.223000,756.290000


In [26]:
df_info = pd.read_csv('Gapminder-info.csv', sep=',', index_col=0)

In [27]:
df_info

,Min,Max,Mid,LogScale,Meaning
Year,1900,2020,1960,False,NaN
Income,300,180000,8000,True,"Per person (GDP/capita, PPP$ inflation-adjusted)"
Life expectancy,0,90,45,False,Years
Population,15000,2000000000,5000000,True,Total
Fertility,0,10,5,False,Babies per woman
Child mortality,1,800,40,True,0-5 year-olds dying per 1000 born


In [28]:
df = df.sort_values(['Year', 'Population'], ascending=[True, False])

In [29]:
df.head()

,Country,Region,Year,Income,Life expectancy,Population,Fertility,Child mortality
4114,China,Asia,1900,776.0,31.98,401579661.0,5.500,416.53
9025,India,Asia,1900,797.0,18.42,291979136.0,5.726,536.09
21335,United States,Americas,1900,6252.0,48.95,78763706.0,3.853,231.70
16381,Russia,Europe,1900,3087.0,30.75,64946671.0,7.360,409.33
7502,Germany,Europe,1900,6029.0,43.94,55185341.0,4.932,372.17


## Create the bubble chart

In [30]:
marker_dict = dict(
    opacity=1,
    line=dict(
        color='#333',
        width=0.4
    )
)
layout_dict = dict(
    plot_bgcolor='white',
    font=dict(color='#333')
)
axes_dict = dict(
    gridcolor='lightgray',
    showline=True,
    linecolor='dimgray',
    linewidth=1,
    showspikes=True,
    spikethickness=0.4,
    spikecolor='dimgray'
)

## A few things to increase similarity

The cell below already runs, but the chart it produces looks nothing like Gapminder — small bubbles, linear x axis, cluttered hover, no styling.

Four changes will get us most of the way there:

- **Larger bubbles** — `size_max` controls the maximum bubble radius.
- **Log scale on the x axis** — `log_x` makes the income axis logarithmic.
- **Reduce hover clutter** — pass a `hover_data` dict that turns off most columns.
- **Style markers, layout, and axes** — apply the `marker_dict`, `layout_dict`, and `axes_dict` we just defined above.

**TODO:** fix each blank below. Run the cell as you go to see the chart improve.

In [31]:
fig = px.scatter(
    df.query('Year==2020'),
    x='Income',
    y='Life expectancy',
    color='Region',
    size='Population',
    hover_name='Country',
    title='Gapminder<br><sup>Data by gapminder.org, CC-BY license</sup>',
    # Make the largest bubbles roughly 60 px
    size_max=60,
    # Put the x axis on a log scale
    log_x=True,
    # Hide every column from the hover tooltip
    # (keep only the country name, which is set by hover_name above)
    hover_data={
        'Region':False,
        'Income':False,
        'Life expectancy':False,
        'Population':False,
    }
)

fig.update_traces(marker=marker_dict)

fig.update_layout(layout_dict)

fig.update_xaxes(axes_dict)
fig.update_yaxes(axes_dict)

fig.show()


## Add animation

### Step 1 — Basic animation

The cell below runs, but it shows all years overlaid on top of each other instead of animating. `px.scatter` can build the animation for you almost for free — two arguments do the work:

- `animation_frame` — the column whose values become the *frames* of the animation (one frame per unique value).
- `animation_group` — the column that identifies the *same entity across frames*, so Plotly knows which bubble is which when it transitions between frames.

**TODO:** fill in the two blanks below so the chart animates over time, with one bubble per country.

In [32]:
fig = px.scatter(
    df.query('Year >= 1800'),
    x='Income',
    range_x=[0, 128000],
    y='Life expectancy',
    range_y=[1, 100],
    color='Region',
    size='Population',
    hover_name='Country',
    size_max=60,
    title='Gapminder<br><sup>Data by gapminder.org, CC-BY license</sup>',
    # TODO: which column should each frame of the animation correspond to?
    animation_frame='Year',
    # TODO: which column identifies the same entity (one bubble) across frames?
    animation_group='Country',
    hover_data={
        'Region':False,
        'Income':False,
        'Life expectancy':False,
        'Population':False,
        'Year':False,
    }
)

fig.update_traces(marker=marker_dict)

fig.update_layout(layout_dict)

fig.update_xaxes(axes_dict)
fig.update_yaxes(axes_dict)

fig.show()


### Step 2 — Add the year as a faded background number

The original Gapminder tool shows the current year as a giant pale number behind the bubbles — it changes as the animation plays. We'll do the same.

The cells below already run end-to-end (the function returns a working animated chart), but the background year is invisible because the placeholder traces are empty. Your job is to fill in the blanks so the year actually appears and updates with each frame.

This needs two ingredients:

1. A helper `background_year(...)` that builds a `go.Scatter` trace with a single text point in the middle of the chart.
2. Adding that trace to **every frame** of the animation, not just the first view, so the year updates as the animation plays.

In Plotly, an animated figure has a list of `fig.frames`, each with its own `frame.data` (the traces shown in that frame). To make the year update, we need to put a fresh background-year trace at the start of each frame's data.

We'll also wrap everything in a `gapminder_fig(xaxis, yaxis)` function so we can reuse it later for the Dash app.

In [33]:
def background_year(year, xaxis, yaxis):
    """Return a go.Scatter trace showing `year` as huge faded text in the chart center."""
    return go.Scatter(
        # TODO: place a single point at the middle of both axes
        # (use the 'Mid' column of df_info)
        x=[df_info.loc[xaxis, 'Mid']],
        y=[df_info.loc[yaxis, 'Mid']],
        mode='text',
        # TODO: the text shown should be the year, as a string
        text=[str(year)],
        showlegend=False,
        textfont=dict(size=200, color='lightgray'),
        textposition='middle center'
    )


In [ ]:
def gapminder_fig(xaxis='Income', yaxis='Life expectancy'):
    fig = px.scatter(
        df.query('Year >= 1800'),
        x=xaxis,
        y=yaxis,
        color='Region',
        size='Population',
        hover_name='Country',
        size_max=60,
        log_x=df_info.loc[xaxis, 'LogScale'],
        log_y=df_info.loc[yaxis, 'LogScale'],
        hover_data={c: False for c in df.columns},
        title='Gapminder<br><sup>Data by gapminder.org, CC-BY license</sup>',
        animation_frame='Year',
        animation_group='Country',
        range_x=[df_info.loc[xaxis, 'Min'], df_info.loc[xaxis, 'Max']],
        range_y=[df_info.loc[yaxis, 'Min'], df_info.loc[yaxis, 'Max']]
    )

    fig.update_traces(marker=marker_dict)
    fig.update_layout(layout_dict)
    fig.update_xaxes(axes_dict)
    fig.update_yaxes(axes_dict)

    # --- Add the background-year trace ---
    # (a) Add it to the initial view (fig.data) so it shows before play is pressed.
    # The first frame's name is the starting year.
    fig.data.
    frame_year = fig.frames[0].name
    # TODO: call background_year(...) with the right arguments and add it as a trace
    fig.add_trace(background_year(frame_year, xaxis, yaxis))
    # Move the new trace to the BACK so bubbles render on top of it.
    fig.data = (fig.data[-1],) + fig.data[:-1]

    # (b) Add a fresh background-year trace to every frame, so the number updates
    # as the animation plays. Each frame.name holds that frame's year.
    for frame in fig.frames:
        # TODO: prepend a background_year trace for THIS frame's year to frame.data
        frame.data = (background_year(frame.name, xaxis, yaxis),) + frame.data
    fig.update(frames=fig.frames)

    # Axis annotations
    fig.add_annotation(
        x=1, y=0, xref='x domain', yref='y domain',
        text=df_info.loc[xaxis, 'Meaning'],
        showarrow=False, align='right'
    )
    fig.add_annotation(
        x=0, y=1, xref='x domain', yref='y domain',
        text=df_info.loc[yaxis, 'Meaning'],
        showarrow=False, valign='top', textangle=-90
    )

    return fig

fig.show()


## Use Dash

- Dash html components: https://dash.plotly.com/dash-html-components
- Dash core components: https://dash.plotly.com/dash-core-components

In [35]:
from dash import Dash, html, dcc, callback, Output, Input

In [36]:
import dash
print(dash.__version__)

4.1.0


In [37]:
import json

In [38]:
attributes = ['Income', 'Life expectancy', 'Fertility', 'Child mortality']
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = Dash(__name__, external_stylesheets=external_stylesheets)

## Let's add interactivity. 
Look at the callback and change the update function to ensure it adapts to user inputs.

## Putting it all together
Ensure that your app runs and displays Gapminder data in the style of Gapminder!

In [39]:
app.layout = html.Div([
    html.H1('Interactive data visualization', 
            style={'fontSize': 50, 'color': 'brown'}),
    html.P('X axis'),
    dcc.Dropdown(
        id='dropdown_x',
        options=[{'label': a, 'value': a} for a in attributes],
        value='Income'
    ),
    html.P('Y axis'),
    dcc.Dropdown(
        id='dropdown_y',
        options=[{'label': a, 'value': a} for a in attributes],
        value='Life expectancy'
    ),
    dcc.Graph(
        id='plot',
        figure=gapminder_fig()
    ),
    dcc.Textarea(
        id='text',
        value=''
    )
])

#TODO: fix this callback to get proper interactivity (framing)
@app.callback(
    Output('plot', 'figure'),
    [Input('dropdown_x', 'value'), Input('dropdown_y', 'value')]
)
def update_plot(x, y):
    return gapminder_fig('Life Expectancy', 'Income')

@app.callback(
    Output('text', 'value'),
    Input('plot', 'selectedData')
)
def print_value(selected_data):
    return json.dumps(selected_data, indent=2)

In [40]:
app.run(port=2218)